In [0]:
import json
from pyspark.sql.functions import col, current_timestamp

# ---------------------------------------------------------
# 1. VOLUME & PATH CONFIGURATION (Unity Catalog)
# ---------------------------------------------------------
BRONZE_VOLUME_PATH = "/Volumes/proj_databricks/uber_medallion_layers/bronze"

# ---------------------------------------------------------
# 2. AZURE EVENT HUBS CONFIGURATION (KAFKA PROTOCOL)
# ---------------------------------------------------------
RAW_CONN_STR = "Endpoint=sb://uber-pipeline-2026.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=BHNHVl7BHs/A7DYcmOh1KFlPFOFzujevx+AEhPsWEu8="

# Kafka JAAS config for Event Hubs Authentication
BOOTSTRAP_SERVERS = "uber-pipeline-2026.servicebus.windows.net:9093"
EH_SASL_JAAS = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{RAW_CONN_STR}";'

# Common Kafka Config Map
kafka_base_options = {
    "kafka.bootstrap.servers": BOOTSTRAP_SERVERS,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": EH_SASL_JAAS,
    "startingOffsets": "earliest"
}

# ---------------------------------------------------------
# 3. READ LIVE STREAMS FROM EVENT HUBS (KAFKA FORMAT)
# ---------------------------------------------------------
# Stream 1: Driver Locations
df_driver_raw = (spark.readStream
    .format("kafka")
    .options(**kafka_base_options)
    .option("subscribe", "eh-driver-locations")
    .load()
)

# Stream 2: Ride Events
df_ride_raw = (spark.readStream
    .format("kafka")
    .options(**kafka_base_options)
    .option("subscribe", "eh-ride-events")
    .load()
)

# ---------------------------------------------------------
# 4. WRITE RAW STREAMS TO DELTA LAKE IN BRONZE VOLUME
# ---------------------------------------------------------
# Stream 1: Driver Locations Stream
query_driver = (df_driver_raw
    .select(
        col("value").cast("string").alias("raw_payload"),
        col("timestamp").alias("event_enqueued_time"),
        current_timestamp().alias("ingestion_timestamp")
    )
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{BRONZE_VOLUME_PATH}/_checkpoints/bronze_driver_locations")
    .start(f"{BRONZE_VOLUME_PATH}/bronze_driver_locations")
)

# Stream 2: Ride Events Stream
query_ride = (df_ride_raw
    .select(
        col("value").cast("string").alias("raw_payload"),
        col("timestamp").alias("event_enqueued_time"),
        current_timestamp().alias("ingestion_timestamp")
    )
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{BRONZE_VOLUME_PATH}/_checkpoints/bronze_ride_events")
    .start(f"{BRONZE_VOLUME_PATH}/bronze_ride_events")
)

In [0]:
#  Check Driver Locations Delta Data
display(
    spark.read.format("delta")
    .load("/Volumes/proj_databricks/uber_medallion_layers/bronze/bronze_driver_locations")
    .orderBy(col("ingestion_timestamp").desc())
)

raw_payload,event_enqueued_time,ingestion_timestamp
"{""driver_id"": ""D104"", ""city"": ""Delhi"", ""latitude"": 28.634011, ""longitude"": 77.218891, ""status"": ""On_Trip"", ""timestamp"": ""2026-08-01T08:18:38Z""}",2026-08-01T08:19:14.822Z,2026-08-01T08:19:15.306Z
"{""driver_id"": ""D105"", ""city"": ""Mumbai"", ""latitude"": 19.081439, ""longitude"": 72.882311, ""status"": ""Available"", ""timestamp"": ""2026-08-01T08:18:36Z""}",2026-08-01T08:19:11.402Z,2026-08-01T08:19:11.731Z
"{""driver_id"": ""D118"", ""city"": ""Pune"", ""latitude"": 18.510997, ""longitude"": 73.871512, ""status"": ""On_Trip"", ""timestamp"": ""2026-08-01T08:18:32Z""}",2026-08-01T08:19:08.901Z,2026-08-01T08:19:09.118Z
"{""driver_id"": ""D101"", ""city"": ""Mumbai"", ""latitude"": 19.080457, ""longitude"": 72.876803, ""status"": ""On_Trip"", ""timestamp"": ""2026-08-01T08:18:29Z""}",2026-08-01T08:19:04.933Z,2026-08-01T08:19:05.044Z
"{""driver_id"": ""D117"", ""city"": ""Pune"", ""latitude"": 18.502547, ""longitude"": 73.844739, ""status"": ""On_Trip"", ""timestamp"": ""2026-08-01T08:18:27Z""}",2026-08-01T08:19:01.976Z,2026-08-01T08:19:02.047Z
"{""driver_id"": ""D115"", ""city"": ""Bangalore"", ""latitude"": 12.972089, ""longitude"": 77.572782, ""status"": ""Busy"", ""timestamp"": ""2026-08-01T08:18:23Z""}",2026-08-01T08:18:59.517Z,2026-08-01T08:18:59.603Z
"{""driver_id"": ""D111"", ""city"": ""Pune"", ""latitude"": 18.508339, ""longitude"": 73.858582, ""status"": ""Busy"", ""timestamp"": ""2026-08-01T08:18:18Z""}",2026-08-01T08:18:55.703Z,2026-08-01T08:18:55.903Z
"{""driver_id"": ""D112"", ""city"": ""Delhi"", ""latitude"": 28.607404, ""longitude"": 77.19226, ""status"": ""Busy"", ""timestamp"": ""2026-08-01T07:46:28Z""}",2026-08-01T07:47:03.531Z,2026-08-01T08:18:53.442Z
"{""driver_id"": ""D117"", ""city"": ""Pune"", ""latitude"": 18.513618, ""longitude"": 73.865822, ""status"": ""Busy"", ""timestamp"": ""2026-08-01T07:46:39Z""}",2026-08-01T07:47:14.61Z,2026-08-01T08:18:53.442Z
"{""driver_id"": ""D102"", ""city"": ""Pune"", ""latitude"": 18.523394, ""longitude"": 73.841942, ""status"": ""On_Trip"", ""timestamp"": ""2026-08-01T07:46:42Z""}",2026-08-01T07:47:18.783Z,2026-08-01T08:18:53.442Z


In [0]:
#  Check Ride Events Delta Data
display(
    spark.read.format("delta")
    .load("/Volumes/proj_databricks/uber_medallion_layers/bronze/bronze_ride_events")
    .orderBy(col("ingestion_timestamp").desc())
)

raw_payload,event_enqueued_time,ingestion_timestamp
"{""ride_id"": ""R5113"", ""driver_id"": ""D193"", ""customer_id"": ""C1047"", ""pickup_city"": ""Kolkata"", ""drop_city"": ""Kolkata"", ""fare"": 1741.84, ""distance_km"": 5.6, ""ride_status"": ""In_Progress"", ""payment_method"": ""UPI"", ""timestamp"": ""2026-08-01T08:18:43Z""}",2026-08-01T08:19:19.163Z,2026-08-01T08:19:19.619Z
"{""ride_id"": ""R5114"", ""driver_id"": ""D150"", ""customer_id"": ""C1021"", ""pickup_city"": ""Bangalore"", ""drop_city"": ""Bangalore"", ""fare"": 989.14, ""distance_km"": 17.4, ""ride_status"": ""Requested"", ""payment_method"": ""UPI"", ""timestamp"": ""2026-08-01T08:18:43Z""}",2026-08-01T08:19:19.38Z,2026-08-01T08:19:19.619Z
"{""ride_id"": ""R5111"", ""driver_id"": ""D186"", ""customer_id"": ""C1045"", ""pickup_city"": ""Pune"", ""drop_city"": ""Pune"", ""fare"": 1020.36, ""distance_km"": 24.9, ""ride_status"": ""Completed"", ""payment_method"": ""Card"", ""timestamp"": ""2026-08-01T08:18:43Z""}",2026-08-01T08:19:19.163Z,2026-08-01T08:19:19.619Z
"{""ride_id"": ""R5112"", ""driver_id"": ""D199"", ""customer_id"": ""C1046"", ""pickup_city"": ""Jaipur"", ""drop_city"": ""Jaipur"", ""fare"": 159.56, ""distance_km"": 25.8, ""ride_status"": ""Completed"", ""payment_method"": ""UPI"", ""timestamp"": ""2026-08-01T08:18:43Z""}",2026-08-01T08:19:19.273Z,2026-08-01T08:19:19.619Z
"{""ride_id"": ""R5110"", ""driver_id"": ""D140"", ""customer_id"": ""C1044"", ""pickup_city"": ""Delhi"", ""drop_city"": ""Delhi"", ""fare"": 1526.84, ""distance_km"": 32.0, ""ride_status"": ""Completed"", ""payment_method"": ""Cash"", ""timestamp"": ""2026-08-01T08:18:43Z""}",2026-08-01T08:19:19.413Z,2026-08-01T08:19:19.619Z
"{""ride_id"": ""R5108"", ""driver_id"": ""D196"", ""customer_id"": ""C201"", ""pickup_city"": ""Ahmedabad"", ""drop_city"": ""Ahmedabad"", ""fare"": 478.21, ""distance_km"": 8.4, ""ride_status"": ""Driver_Arrived"", ""payment_method"": ""UPI"", ""timestamp"": ""2026-08-01T08:18:39Z""}",2026-08-01T08:19:14.723Z,2026-08-01T08:19:15.04Z
"{""ride_id"": ""R5107"", ""driver_id"": ""D186"", ""customer_id"": ""C290"", ""pickup_city"": ""Pune"", ""drop_city"": ""Pune"", ""fare"": 1241.53, ""distance_km"": 11.5, ""ride_status"": ""Completed"", ""payment_method"": ""UPI"", ""timestamp"": ""2026-08-01T08:18:39Z""}",2026-08-01T08:19:14.739Z,2026-08-01T08:19:15.04Z
"{""ride_id"": ""R5109"", ""driver_id"": ""D151"", ""customer_id"": ""C214"", ""pickup_city"": ""Bangalore"", ""drop_city"": ""Bangalore"", ""fare"": 1263.28, ""distance_km"": 6.2, ""ride_status"": ""Accepted"", ""payment_method"": ""Cash"", ""timestamp"": ""2026-08-01T08:18:39Z""}",2026-08-01T08:19:14.771Z,2026-08-01T08:19:15.04Z
"{""ride_id"": ""R5106"", ""driver_id"": ""D121"", ""customer_id"": ""C1043"", ""pickup_city"": ""Mumbai"", ""drop_city"": ""Mumbai"", ""fare"": 767.28, ""distance_km"": 21.4, ""ride_status"": ""Completed"", ""payment_method"": ""Cash"", ""timestamp"": ""2026-08-01T08:18:39Z""}",2026-08-01T08:19:14.818Z,2026-08-01T08:19:15.04Z
"{""ride_id"": ""R5104"", ""driver_id"": ""D152"", ""customer_id"": ""C109"", ""pickup_city"": ""Bangalore"", ""drop_city"": ""Bangalore"", ""fare"": 563.34, ""distance_km"": 27.5, ""ride_status"": ""In_Progress"", ""payment_method"": ""Cash"", ""timestamp"": ""2026-08-01T08:18:34Z""}",2026-08-01T08:19:10.551Z,2026-08-01T08:19:10.892Z


In [0]:
#  Check Ride Events Delta Data
display(
    spark.read.format("delta")
    .load("/Volumes/proj_databricks/uber_medallion_layers/bronze/bronze_ride_events/")
)



#  Check Ride Events Delta Data
display(
    spark.read.format("delta")
    .load("/Volumes/proj_databricks/uber_medallion_layers/bronze/bronze_ride_events")
    .orderBy(col("ingestion_timestamp").desc())

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
%sql
DROP VIEW IF EXISTS proj_databricks.uber_medallion_layers.fact_ride_events;

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
# Databricks Python cell me run karo:
dbutils.fs.rm("/Volumes/proj_databricks/uber_medallion_layers/uber_gold/_checkpoints", True)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
# =========================================================
# DEEP CLEAN SCRIPT FOR MEDALLION LAYERS & CHECKPOINTS
# =========================================================

paths_to_clean = [
    "/Volumes/proj_databricks/uber_medallion_layers/bronze",
    "/Volumes/proj_databricks/uber_medallion_layers/uber_silver",
    "/Volumes/proj_databricks/uber_medallion_layers/uber_gold"
]

print("🧹 Starting Deep Clean of Medallion Volumes...\n")

for path in paths_to_clean:
    try:
        # Delete recursively (True)
        dbutils.fs.rm(path, True)
        print(f"✅ Cleaned path successfully: {path}")
    except Exception as e:
        print(f"⚠️ Could not clean {path}: {str(e)}")

print("\n✨ All layers, streaming checkpoints, and delta tables are 100% WIPED OUT!")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
dbutils.fs.mkdirs("/Volumes/proj_databricks/uber_medallion_layers/bronze")
dbutils.fs.mkdirs("/Volumes/proj_databricks/uber_medallion_layers/uber_silver")
dbutils.fs.mkdirs("/Volumes/proj_databricks/uber_medallion_layers/uber_gold")
print("📁 Directories Recreated Successfully!")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data